# Week 3 Assignment: Subqueries, CTEs, and Window Functions
### Superstore Dataset Analysis
**Objective:** Use Subqueries, CTEs, and Window Functions to analyze sales data from the Superstore dataset.

## Setup & Imports

In [1]:
import pandas as pd
import sqlite3
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.2f}".format)
print("Libraries imported successfully!")

Libraries imported successfully!


## Step 1: Setup Data
### Load Superstore CSV and create SQLite tables

In [2]:
# Load CSV
df = pd.read_csv("Sample_-_Superstore.csv", encoding="latin1")

# Clean column names
df.columns = [c.strip().replace(" ", "_").replace("-", "_") for c in df.columns]

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(3)

Dataset shape: (9994, 21)
Columns: ['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']


,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.00,41.91
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.00,219.58
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.00,6.87


In [3]:
# Create in-memory SQLite database
conn = sqlite3.connect(":memory:")

# Insert full data into superstore_raw
df.to_sql("superstore_raw", conn, index=False, if_exists="replace")
print("superstore_raw table created!")

# Create customers table
conn.execute("""
    CREATE TABLE customers AS
    SELECT DISTINCT Customer_ID, Customer_Name, Segment, Country, City, State, Region
    FROM superstore_raw
""")
print("customers table created!")

# Create orders table
conn.execute("""
    CREATE TABLE orders AS
    SELECT DISTINCT Order_ID, Order_Date, Ship_Date, Ship_Mode, Customer_ID, Sales, Quantity, Discount, Profit
    FROM superstore_raw
""")
print("orders table created!")

# Create products table
conn.execute("""
    CREATE TABLE products AS
    SELECT DISTINCT Product_ID, Product_Name, Category, Sub_Category
    FROM superstore_raw
""")
print("products table created!")
conn.commit()

# Verify row counts
for table in ["superstore_raw", "customers", "orders", "products"]:
    count = pd.read_sql("SELECT COUNT(*) as cnt FROM " + table, conn)["cnt"][0]
    print(table + ":", count, "rows")

superstore_raw table created!
customers table created!
orders table created!
products table created!


superstore_raw: 9994 rows
customers: 4688 rows
orders: 9993 rows
products: 1894 rows


In [4]:
# Preview each table
print("=== customers ===")
display(pd.read_sql("SELECT * FROM customers LIMIT 3", conn))
print("\n=== orders ===")
display(pd.read_sql("SELECT * FROM orders LIMIT 3", conn))
print("\n=== products ===")
display(pd.read_sql("SELECT * FROM products LIMIT 3", conn))

=== customers ===


,Customer_ID,Customer_Name,Segment,Country,City,State,Region
0,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,South
1,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,West
2,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,South



=== orders ===


,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Sales,Quantity,Discount,Profit
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,261.96,2,0.00,41.91
1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,731.94,3,0.00,219.58
2,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,14.62,2,0.00,6.87



=== products ===


,Product_ID,Product_Name,Category,Sub_Category
0,FUR-BO-10001798,Bush Somerset Collection Bookcase,Furniture,Bookcases
1,FUR-CH-10000454,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Furniture,Chairs
2,OFF-LA-10000240,Self-Adhesive Address Labels for Typewriters b...,Office Supplies,Labels


## Step 2: Perform Required Queries

### Q1. Orders where Sales > Average Sales *(Subquery)*

In [5]:
query1 = """
SELECT Order_ID, Customer_ID, ROUND(Sales, 2) AS Sales
FROM orders
WHERE Sales > (
    SELECT AVG(Sales) FROM orders
)
ORDER BY Sales DESC
LIMIT 10
"""
result1 = pd.read_sql(query1, conn)
avg_sales = pd.read_sql("SELECT ROUND(AVG(Sales),2) as avg FROM orders", conn)["avg"][0]
print("Average Sales: $", avg_sales)
print("Orders above average:", pd.read_sql("SELECT COUNT(*) as c FROM orders WHERE Sales > (SELECT AVG(Sales) FROM orders)", conn)["c"][0])
print("\nTop 10 results:")
result1

Average Sales: $ 229.85
Orders above average: 2359

Top 10 results:


,Order_ID,Customer_ID,Sales
0,CA-2014-145317,SM-20320,22638.48
1,CA-2016-118689,TC-20980,17499.95
2,CA-2017-140151,RB-19360,13999.96
3,CA-2017-127180,TA-21385,11199.97
4,CA-2017-166709,HL-15040,10499.97
5,CA-2016-117121,AB-10105,9892.74
6,CA-2014-116904,SC-20095,9449.95
7,US-2016-107440,BS-11365,9099.93
8,CA-2016-158841,SE-20110,8749.95
9,CA-2016-143714,CC-12370,8399.98


### Q2. Highest Sales Order for Each Customer *(Subquery)*

In [6]:
query2 = """
SELECT o.Customer_ID, c.Customer_Name, o.Order_ID, ROUND(o.Sales, 2) AS Max_Sales
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
WHERE o.Sales = (
    SELECT MAX(o2.Sales)
    FROM orders o2
    WHERE o2.Customer_ID = o.Customer_ID
)
ORDER BY Max_Sales DESC
LIMIT 10
"""
result2 = pd.read_sql(query2, conn)
print("Highest sales order per customer (Top 10):")
result2

Highest sales order per customer (Top 10):


,Customer_ID,Customer_Name,Order_ID,Max_Sales
0,SM-20320,Sean Miller,CA-2014-145317,22638.48
1,SM-20320,Sean Miller,CA-2014-145317,22638.48
2,SM-20320,Sean Miller,CA-2014-145317,22638.48
3,SM-20320,Sean Miller,CA-2014-145317,22638.48
4,SM-20320,Sean Miller,CA-2014-145317,22638.48
5,TC-20980,Tamara Chand,CA-2016-118689,17499.95
6,TC-20980,Tamara Chand,CA-2016-118689,17499.95
7,TC-20980,Tamara Chand,CA-2016-118689,17499.95
8,TC-20980,Tamara Chand,CA-2016-118689,17499.95
9,TC-20980,Tamara Chand,CA-2016-118689,17499.95


### Q3. Total Sales for Each Customer *(CTE)*

In [7]:
query3 = """
WITH customer_sales AS (
    SELECT o.Customer_ID, c.Customer_Name,
           ROUND(SUM(o.Sales), 2) AS Total_Sales
    FROM orders o
    JOIN customers c ON o.Customer_ID = c.Customer_ID
    GROUP BY o.Customer_ID, c.Customer_Name
)
SELECT * FROM customer_sales
ORDER BY Total_Sales DESC
LIMIT 10
"""
result3 = pd.read_sql(query3, conn)
print("Total sales per customer (Top 10):")
result3

Total sales per customer (Top 10):


,Customer_ID,Customer_Name,Total_Sales
0,KL-16645,Ken Lonsdale,141752.29
1,SE-20110,Sanjit Engle,134303.82
2,AB-10105,Adrian Barton,130262.14
3,SM-20320,Sean Miller,125215.25
4,CL-12565,Clay Ludtke,119686.01
5,SC-20095,Sanjit Chand,113138.67
6,SV-20365,Seth Vernon,103238.55
7,ZC-21910,Zuschuss Carroll,96308.48
8,ME-17320,Maria Etezadi,95973.55
9,LA-16780,Laura Armstrong,95405.44


### Q4. Customers with Total Sales Above Average *(CTE + Subquery)*

In [8]:
query4 = """
WITH customer_sales AS (
    SELECT o.Customer_ID, c.Customer_Name,
           ROUND(SUM(o.Sales), 2) AS Total_Sales
    FROM orders o
    JOIN customers c ON o.Customer_ID = c.Customer_ID
    GROUP BY o.Customer_ID, c.Customer_Name
)
SELECT *
FROM customer_sales
WHERE Total_Sales > (
    SELECT AVG(Total_Sales) FROM customer_sales
)
ORDER BY Total_Sales DESC
"""
result4 = pd.read_sql(query4, conn)
print("Customers above average total sales:", len(result4))
result4.head(10)

Customers above average total sales: 278


,Customer_ID,Customer_Name,Total_Sales
0,KL-16645,Ken Lonsdale,141752.29
1,SE-20110,Sanjit Engle,134303.82
2,AB-10105,Adrian Barton,130262.14
3,SM-20320,Sean Miller,125215.25
4,CL-12565,Clay Ludtke,119686.01
5,SC-20095,Sanjit Chand,113138.67
6,SV-20365,Seth Vernon,103238.55
7,ZC-21910,Zuschuss Carroll,96308.48
8,ME-17320,Maria Etezadi,95973.55
9,LA-16780,Laura Armstrong,95405.44


### Q5. Rank All Customers by Total Sales *(Window Function)*

In [9]:
query5 = """
SELECT Customer_ID, Customer_Name, Total_Sales,
       RANK() OVER (ORDER BY Total_Sales DESC) AS Sales_Rank
FROM (
    SELECT o.Customer_ID, c.Customer_Name,
           ROUND(SUM(o.Sales), 2) AS Total_Sales
    FROM orders o
    JOIN customers c ON o.Customer_ID = c.Customer_ID
    GROUP BY o.Customer_ID, c.Customer_Name
)
ORDER BY Sales_Rank
LIMIT 10
"""
result5 = pd.read_sql(query5, conn)
print("Customer Sales Ranking (Top 10):")
result5

Customer Sales Ranking (Top 10):


,Customer_ID,Customer_Name,Total_Sales,Sales_Rank
0,KL-16645,Ken Lonsdale,141752.29,1
1,SE-20110,Sanjit Engle,134303.82,2
2,AB-10105,Adrian Barton,130262.14,3
3,SM-20320,Sean Miller,125215.25,4
4,CL-12565,Clay Ludtke,119686.01,5
5,SC-20095,Sanjit Chand,113138.67,6
6,SV-20365,Seth Vernon,103238.55,7
7,ZC-21910,Zuschuss Carroll,96308.48,8
8,ME-17320,Maria Etezadi,95973.55,9
9,LA-16780,Laura Armstrong,95405.44,10


### Q6. Row Numbers per Order within Each Customer *(Window Function + PARTITION BY)*

In [10]:
query6 = """
SELECT o.Customer_ID, c.Customer_Name, o.Order_ID, o.Order_Date,
       ROUND(o.Sales, 2) AS Sales,
       ROW_NUMBER() OVER (PARTITION BY o.Customer_ID ORDER BY o.Order_Date) AS Order_Row_Num
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
ORDER BY o.Customer_ID, Order_Row_Num
LIMIT 15
"""
result6 = pd.read_sql(query6, conn)
print("Row numbers per order within each customer (first 15 rows):")
result6

Row numbers per order within each customer (first 15 rows):


,Customer_ID,Customer_Name,Order_ID,Order_Date,Sales,Order_Row_Num
0,AA-10315,Alex Avila,CA-2015-121391,10/4/2015,26.96,1
1,AA-10315,Alex Avila,CA-2015-121391,10/4/2015,26.96,2
2,AA-10315,Alex Avila,CA-2015-121391,10/4/2015,26.96,3
3,AA-10315,Alex Avila,CA-2015-121391,10/4/2015,26.96,4
4,AA-10315,Alex Avila,CA-2016-103982,3/3/2016,3930.07,5
5,AA-10315,Alex Avila,CA-2016-103982,3/3/2016,3930.07,6
6,AA-10315,Alex Avila,CA-2016-103982,3/3/2016,3930.07,7
7,AA-10315,Alex Avila,CA-2016-103982,3/3/2016,3930.07,8
8,AA-10315,Alex Avila,CA-2016-103982,3/3/2016,2.30,9
9,AA-10315,Alex Avila,CA-2016-103982,3/3/2016,2.30,10


### Q7. Top 3 Customers by Total Sales *(Window Function)*

In [11]:
query7 = """
SELECT * FROM (
    SELECT o.Customer_ID, c.Customer_Name,
           ROUND(SUM(o.Sales), 2) AS Total_Sales,
           DENSE_RANK() OVER (ORDER BY SUM(o.Sales) DESC) AS Sales_Rank
    FROM orders o
    JOIN customers c ON o.Customer_ID = c.Customer_ID
    GROUP BY o.Customer_ID, c.Customer_Name
)
WHERE Sales_Rank <= 3
"""
result7 = pd.read_sql(query7, conn)
print("Top 3 Customers by Total Sales:")
result7

Top 3 Customers by Total Sales:


,Customer_ID,Customer_Name,Total_Sales,Sales_Rank
0,KL-16645,Ken Lonsdale,141752.29,1
1,SE-20110,Sanjit Engle,134303.82,2
2,AB-10105,Adrian Barton,130262.14,3


## Step 3: Final Combined Query
*(JOIN + CTE + Window Function)*

In [12]:
query_final = """
WITH customer_totals AS (
    SELECT
        o.Customer_ID,
        c.Customer_Name,
        ROUND(SUM(o.Sales), 2) AS Total_Sales
    FROM orders o
    JOIN customers c ON o.Customer_ID = c.Customer_ID
    GROUP BY o.Customer_ID, c.Customer_Name
),
ranked AS (
    SELECT
        Customer_ID,
        Customer_Name,
        Total_Sales,
        RANK() OVER (ORDER BY Total_Sales DESC) AS Sales_Rank
    FROM customer_totals
)
SELECT Customer_Name, Total_Sales, Sales_Rank
FROM ranked
ORDER BY Sales_Rank
LIMIT 15
"""
result_final = pd.read_sql(query_final, conn)
print("Final Combined Query — Customer Name | Total Sales | Rank (Top 15):")
result_final

Final Combined Query — Customer Name | Total Sales | Rank (Top 15):


,Customer_Name,Total_Sales,Sales_Rank
0,Ken Lonsdale,141752.29,1
1,Sanjit Engle,134303.82,2
2,Adrian Barton,130262.14,3
3,Sean Miller,125215.25,4
4,Clay Ludtke,119686.01,5
5,Sanjit Chand,113138.67,6
6,Seth Vernon,103238.55,7
7,Zuschuss Carroll,96308.48,8
8,Maria Etezadi,95973.55,9
9,Laura Armstrong,95405.44,10


## Mini Project: Customer Sales Insights

### 1. Top 5 Customers

In [13]:
top5 = """
WITH cs AS (
    SELECT o.Customer_ID, c.Customer_Name, ROUND(SUM(o.Sales),2) AS Total_Sales
    FROM orders o JOIN customers c ON o.Customer_ID = c.Customer_ID
    GROUP BY o.Customer_ID, c.Customer_Name
)
SELECT Customer_Name, Total_Sales,
       RANK() OVER (ORDER BY Total_Sales DESC) AS Rank
FROM cs ORDER BY Total_Sales DESC LIMIT 5
"""
print("Top 5 Customers:")
pd.read_sql(top5, conn)

Top 5 Customers:


,Customer_Name,Total_Sales,Rank
0,Ken Lonsdale,141752.29,1
1,Sanjit Engle,134303.82,2
2,Adrian Barton,130262.14,3
3,Sean Miller,125215.25,4
4,Clay Ludtke,119686.01,5


### 2. Bottom 5 Customers

In [14]:
bottom5 = """
WITH cs AS (
    SELECT o.Customer_ID, c.Customer_Name, ROUND(SUM(o.Sales),2) AS Total_Sales
    FROM orders o JOIN customers c ON o.Customer_ID = c.Customer_ID
    GROUP BY o.Customer_ID, c.Customer_Name
)
SELECT Customer_Name, Total_Sales,
       RANK() OVER (ORDER BY Total_Sales ASC) AS Rank_From_Bottom
FROM cs ORDER BY Total_Sales ASC LIMIT 5
"""
print("Bottom 5 Customers:")
pd.read_sql(bottom5, conn)

Bottom 5 Customers:


,Customer_Name,Total_Sales,Rank_From_Bottom
0,Lela Donovan,5.30,1
1,Thais Sissman,9.67,2
2,Carl Jackson,16.52,3
3,Mitch Gastineau,16.74,4
4,Roy Skaria,44.66,5


### 3. Customers Who Made Only One Order

In [15]:
one_order = """
SELECT c.Customer_Name, COUNT(DISTINCT o.Order_ID) AS Order_Count
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
GROUP BY o.Customer_ID, c.Customer_Name
HAVING COUNT(DISTINCT o.Order_ID) = 1
ORDER BY c.Customer_Name
"""
result_one = pd.read_sql(one_order, conn)
print("Customers with exactly 1 order:", len(result_one))
result_one.head(10)

Customers with exactly 1 order: 12


,Customer_Name,Order_Count
0,Anemone Ratner,1
1,Anthony O'Donnell,1
2,Carl Jackson,1
3,Jenna Caffey,1
4,Jocasta Rupert,1
5,Lela Donovan,1
6,Mitch Gastineau,1
7,Patricia Hirasaki,1
8,Ricardo Emerson,1
9,Roland Murray,1


### 4. Customers with Above-Average Sales

In [16]:
above_avg = """
WITH cs AS (
    SELECT o.Customer_ID, c.Customer_Name, ROUND(SUM(o.Sales),2) AS Total_Sales
    FROM orders o JOIN customers c ON o.Customer_ID = c.Customer_ID
    GROUP BY o.Customer_ID, c.Customer_Name
)
SELECT Customer_Name, Total_Sales
FROM cs
WHERE Total_Sales > (SELECT AVG(Total_Sales) FROM cs)
ORDER BY Total_Sales DESC
"""
result_above = pd.read_sql(above_avg, conn)
print("Customers with above-average sales:", len(result_above))
result_above.head(10)

Customers with above-average sales: 278


,Customer_Name,Total_Sales
0,Ken Lonsdale,141752.29
1,Sanjit Engle,134303.82
2,Adrian Barton,130262.14
3,Sean Miller,125215.25
4,Clay Ludtke,119686.01
5,Sanjit Chand,113138.67
6,Seth Vernon,103238.55
7,Zuschuss Carroll,96308.48
8,Maria Etezadi,95973.55
9,Laura Armstrong,95405.44


### 5. Highest Order Value Per Customer

In [17]:
max_order = """
SELECT c.Customer_Name, o.Order_ID, ROUND(o.Sales, 2) AS Max_Order_Sales
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
WHERE o.Sales = (
    SELECT MAX(o2.Sales)
    FROM orders o2
    WHERE o2.Customer_ID = o.Customer_ID
)
ORDER BY Max_Order_Sales DESC
LIMIT 10
"""
print("Highest order value per customer (Top 10):")
pd.read_sql(max_order, conn)

Highest order value per customer (Top 10):


,Customer_Name,Order_ID,Max_Order_Sales
0,Sean Miller,CA-2014-145317,22638.48
1,Sean Miller,CA-2014-145317,22638.48
2,Sean Miller,CA-2014-145317,22638.48
3,Sean Miller,CA-2014-145317,22638.48
4,Sean Miller,CA-2014-145317,22638.48
5,Tamara Chand,CA-2016-118689,17499.95
6,Tamara Chand,CA-2016-118689,17499.95
7,Tamara Chand,CA-2016-118689,17499.95
8,Tamara Chand,CA-2016-118689,17499.95
9,Tamara Chand,CA-2016-118689,17499.95


## Summary
| Concept | Queries |
|---|---|
| **Subquery** | Q1, Q2, Q4, Mini Q5 |
| **CTE** | Q3, Q4, Final, Mini Q1–Q4 |
| **RANK() Window Function** | Q5, Q7, Final, Mini Q1–Q2 |
| **ROW_NUMBER() + PARTITION BY** | Q6 |
| **DENSE_RANK()** | Q7 |
| **JOIN** | Q2–Q7, Final, All Mini |

> All queries use SQLite in-memory database via Python + pandas.